In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# set the parent directory to 2 directories up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
sys.path.append(parent_dir)

In [3]:
import pandas as pd
import numpy as np
from utils.common import list_remove_value, render_and_save_table
import src.regression_analysis.utils.regression_utils as rg


from constants.mappers import LONG_TO_SHORT_MAPPER, SHORT_TO_LONG_MAPPER, COVARIATE_MAPPING

from src.GLOBAL import (
    PROCESSED_COUNTY_LEVEL_MERGED_DATA_FILES,
    SUITABILITY_SCORES_DATA_FILES,
)

pd.set_option('display.max_columns', None)

# create directory if it doesn't exist
os.makedirs('latex', exist_ok=True)

Root directory set to: /Users/jack/Documents/Projects/Power-Lab/Solar NIMBY/Solar-NIMBY-Analysis


In [4]:
factors = pd.read_csv(PROCESSED_COUNTY_LEVEL_MERGED_DATA_FILES['merged_decennial_all'], dtype={"GEOID": str, "STATEFP": str, "COUNTYFP": str, 'Median Income': float})

county_suitability = pd.read_csv(SUITABILITY_SCORES_DATA_FILES['county_suitability_scores'])

merged = factors.merge(county_suitability, on=['State', 'County Name'])

In [5]:
merged

,GEOID,STATEFP,COUNTYFP,State,County Name,area km2,area mi2,Wind Capacity Intensity (MW/ 1000 sq mile),Wind Project Intensity (Projects/ 1000 sq mile),Wind Avg Capacity Intensity (MW/ 1000 sq mile),GDP_2017,GDP_2018,GDP_2019,GDP_2020,GDP_2021,GDP_2022,Solar Capacity Intensity (MW/ 1000 sq mile),Solar Project Intensity (Projects/ 1000 sq mile),Solar Avg Capacity Intensity (MW/ 1000 sq mile),Solar Capacity Intensity (MW/ 1000 sq mile)_small,Solar Project Intensity (Projects/ 1000 sq mile)_small,Solar Avg Capacity Intensity (MW/ 1000 sq mile)_small,Solar Capacity Intensity (MW/ 1000 sq mile)_medium,Solar Project Intensity (Projects/ 1000 sq mile)_medium,Solar Avg Capacity Intensity (MW/ 1000 sq mile)_medium,Solar Capacity Intensity (MW/ 1000 sq mile)_large,Solar Project Intensity (Projects/ 1000 sq mile)_large,Solar Avg Capacity Intensity (MW/ 1000 sq mile)_large,No. of Private Schools,Median Income,Total Unemployment,Unemployment Rate,Number of Existing Installs,Total Installed Capacity (kW),Median Installed Capacity (kW),Total Installed Capacity (kW/ 1000 sq mile),Median Installed Capacity (kW/ 1000 sq mile),Number of Existing Installs (Projects/ 1000 sq mile),Hispanic/Latino,White,Black/African American,American Indian/Alaska Native,Asian,Native Hawaiian/Other Pacific Islander,Others,Rural Area Percentage,Urban Area Percentage,democrat_percentage_vote,republican_percentage_vote,green_percentage_vote,libertarian_percentage_vote,other_percentage_vote,18-24 Less than high school graduate,18-24 High school graduate,18-24 Some college or associate's degree,18-24 Bachelor's degree or higher,25+ Less than 9th grade,"25+ 9th to 12th grade, no diploma",25+ High school graduate,"25+ Some college, no degree",25+ Associate's degree,25+ Bachelor's degree,25+ Graduate or professional degree,25+ High school graduate or higher,25+ Bachelor's degree or higher,Electric Commercial Rate,Electric Industrial Rate,Electric Residential Rate,GHI,Protected_Land,Habitat,Slope,Population_Density,Distance_to_Substation,Land_Cover
0,01001,01,001,Alabama,Autauga,1565.322757,604.374247,NaN,NaN,NaN,29.49,29.91,28.96,28.82,28.91,32.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,62660.0,46577.0,2.8,12.0,227901.00,12.75,377085.89,21.10,19.86,0.036000,0.707117,0.193045,0.003129,0.014846,0.000374,0.043823,0.9637,0.0363,0.270184,0.714368,NaN,NaN,0.015448,9.9,33.4,45.4,11.3,2.0,8.4,32.8,19.6,9.1,16.4,11.7,89.6,28.1,0.121895,0.063652,0.135057,20.000000,99.318005,45.290652,57.424907,92.089577,61.387109,61.463846
1,01003,01,003,Alabama,Baldwin,4352.548564,1680.527706,NaN,NaN,NaN,29.96,31.48,33.07,32.88,35.46,36.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,64346.0,183804.0,3.7,35.0,1123484.25,14.50,668530.63,8.63,20.83,0.054736,0.804666,0.077669,0.005570,0.008754,0.000526,0.046284,0.9138,0.0862,0.224090,0.761714,NaN,NaN,0.014196,14.4,39.1,38.4,8.1,2.1,6.9,27.4,21.7,9.5,20.6,11.8,91.0,32.5,0.121895,0.063652,0.135057,20.000000,88.641000,41.260557,91.483860,81.160259,61.061060,57.486901
2,01005,01,005,Alabama,Barbour,2342.545642,904.461557,NaN,NaN,NaN,30.83,31.33,30.87,29.61,30.27,30.17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,36422.0,20639.0,8.6,NaN,NaN,NaN,NaN,NaN,NaN,0.059866,0.439519,0.469809,0.002299,0.004084,0.000000,0.023629,0.9930,0.0070,0.457882,0.534512,NaN,NaN,0.007606,23.2,40.3,33.1,3.5,7.4,16.9,36.7,20.5,7.3,6.7,4.4,75.7,11.2,0.121895,0.063652,0.135057,20.000000,94.051886,29.536965,66.476186,98.447689,50.000000,59.109448
3,01007,01,007,Alabama,Bibb,1622.295670,626.371603,NaN,NaN,NaN,18.48,18.05,20.06,20.91,20.82,20.38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54277.0,18333.0,9.7,0.0,305.00,18.75,486.93,29.93,0.00,0.033194,0.737541,0.196923,0.001749,0.001166,0.000404,0.027632,1.0000,0.0000,0.206983,0.784263,NaN,NaN,0.008755,14.1,64.3,20.1,1.6,6.2,13.3,43.9,18.0,6.7,7.9,4.0,80.5,11.9,0.121895,0.063652,0.135057,19.892096,88.980580,27.360418,40.416651,95.748793,51.124790,56.760850
4,01009,01,009,Alabama,Blount,1685.098070,65

# Loading Data

In this section we load the preprocessed data, and create a few dataframes with different subsets of the data for analysis.

## Prepare Data 

- `base_data`: Contains all base social factors and techno-economic factors.
- `urban_data`: Includes urban form and land use factors.
- `electric_price_data`: Includes electric price factors.
- `gdp_data`: Includes GDP factors.
- `rooftop_data`: Includes rooftop solar factors.

In [ ]:
from utils.regression_data_prep import prepare_regression_data
from constants.county_table_variables import (
    COLS_TO_NORMALIZE, # Columns to normalize
    # Data column subsets
    BASE_SOCIAL_FACTORS_COLUMNS,
    URBAN_COLUMNS,
    ELECTRIC_PRICE_COLUMNS,
    GDP_COLUMNS,
    ROOFTOP_COLUMNS
)

base_data = prepare_regression_data(merged, BASE_SOCIAL_FACTORS_COLUMNS, COLS_TO_NORMALIZE)

urban_data = prepare_regression_data(merged, URBAN_COLUMNS, COLS_TO_NORMALIZE, columns_to_unnormalized={"Solar Avg Capacity Intensity (MW/ 1000 sq mile)" : "area mi2"})

electric_price_data = prepare_regression_data(merged, ELECTRIC_PRICE_COLUMNS, COLS_TO_NORMALIZE, drop_subset=['Electric Residential Rate', 'Electric Commercial Rate', 'Electric Industrial Rate'])

gdp_data = prepare_regression_data(merged, GDP_COLUMNS, COLS_TO_NORMALIZE, drop_subset=['GDP_2022'])

rooftop_data = prepare_regression_data(merged, ROOFTOP_COLUMNS, COLS_TO_NORMALIZE, drop_subset=['Number of Existing Installs (Projects/ 1000 sq mile)'])


## Subsets of Prepped Data

- `urban_50_data`: Urban areas with at least 50% urban land use.
- `mixture_50_data`: Areas with 10%-50% urban land use.
- `urban_10_data`: Rural areas with less than 10% urban land use.

In [7]:
# Filter rows where 'urban' is greater than or equal to 50
urban_50_data = urban_data[urban_data["Urban Area Percentage"] >= 50]
mixture_50_data = urban_data[(urban_data["Urban Area Percentage"] >= 10) & (urban_data["Urban Area Percentage"] < 50)]
# Filter rows where 'urban' is less than 10
urban_10_data = urban_data[urban_data["Urban Area Percentage"] < 10]

#  Regression

Section contains the different regression models that were used in the analysis, with the different data subsets and the different dependent variables (technoeconomic variables)

In [8]:
from regression import run_base_regressions, create_regression_table
from constants.regression_variables import REGRESSION_VARIABLES, COVARIATE_ORDER

## Base Factors Regression

In [9]:
data = base_data.rename(columns=LONG_TO_SHORT_MAPPER)
independent_vars = data[REGRESSION_VARIABLES["base"]]
techno_base_dict = run_base_regressions(data, independent_vars)

techno_base = (
    techno_base_dict["y1"]["stargazer"].models
    + techno_base_dict["y2"]["stargazer"].models
    + techno_base_dict["y3"]["stargazer"].models
)

## Include Electric Price Regression

In [10]:
data = electric_price_data.rename(columns=LONG_TO_SHORT_MAPPER)
independent_vars = data[REGRESSION_VARIABLES["base"] + REGRESSION_VARIABLES["electric"]]
techno_electric_dict = run_base_regressions(data, independent_vars)

techno05 = (
    techno_electric_dict["y1"]["stargazer"].models
    + techno_electric_dict["y2"]["stargazer"].models
    + techno_electric_dict["y3"]["stargazer"].models
)

## Include Traditional Social Factors Regression

In [11]:
data = base_data.rename(columns=LONG_TO_SHORT_MAPPER)
independent_vars = data[REGRESSION_VARIABLES["base"] + REGRESSION_VARIABLES["social_traditional"]]
trad_social_dict = run_base_regressions(data, independent_vars)

trad_social = (
    trad_social_dict["y1"]["stargazer"].models
    + trad_social_dict["y2"]["stargazer"].models
    + trad_social_dict["y3"]["stargazer"].models
)

## Include most Social Factors Regression

In [12]:
data = gdp_data.rename(columns=LONG_TO_SHORT_MAPPER)
independent_vars = data[REGRESSION_VARIABLES["base"] + REGRESSION_VARIABLES["social_traditional"] + REGRESSION_VARIABLES["social_expanded"]]
social_expanded_dict = run_base_regressions(data, independent_vars)

## Solar Rooftop Factors Regression

In [13]:
data = rooftop_data.rename(columns=LONG_TO_SHORT_MAPPER)
independent_vars = data[
    REGRESSION_VARIABLES["base"]
    + REGRESSION_VARIABLES["social_traditional"]
    + list_remove_value(REGRESSION_VARIABLES["social_expanded"], "GDPpercapita")
]
y1_rooftop_dict = rg.reg_state_fixed('roof_capacity', independent_vars, data)

## Comparing Base Social with different Solar Project Sizes

In [14]:
data = base_data.rename(columns=LONG_TO_SHORT_MAPPER)
independent_vars = data[
    REGRESSION_VARIABLES["base"]
    + REGRESSION_VARIABLES["social_traditional"]
    + list_remove_value(REGRESSION_VARIABLES["social_expanded"], "GDPpercapita")
]

y3_social_het_dict = rg.reg_state_fixed(
    "y3", independent_vars, data
)
y3sm_social_het_dict = rg.reg_state_fixed(
    "y3_small", independent_vars, data
)
y3med_social_het_dict = rg.reg_state_fixed(
    "y3_med", independent_vars, data
)
y3lg_social_het_dict = rg.reg_state_fixed(
    "y3_lg", independent_vars, data
)


techno_base_size = (
    y3_social_het_dict["stargazer"].models
    + y3sm_social_het_dict["stargazer"].models
    + y3med_social_het_dict["stargazer"].models
    + y3lg_social_het_dict["stargazer"].models
)

## Varying Urban/Rural Composition Regression

In [15]:
urban_50 = urban_50_data.rename(columns=LONG_TO_SHORT_MAPPER)
mixture = mixture_50_data.rename(columns=LONG_TO_SHORT_MAPPER)
urban_10 = urban_10_data.rename(columns=LONG_TO_SHORT_MAPPER)

independent_vars_cols = REGRESSION_VARIABLES["base"] + REGRESSION_VARIABLES["social_traditional"] + list_remove_value(REGRESSION_VARIABLES["social_expanded"], "GDPpercapita")

# urban50
urban_50_independent_vars = urban_50[independent_vars_cols]
y1_urban_50_dict = rg.reg_state_fixed('y1', urban_50_independent_vars, urban_50)
y3_urban_50_dict = rg.reg_state_fixed('y3', urban_50_independent_vars, urban_50)
# mixture50
mixture_independent_vars = mixture[independent_vars_cols]
y1_mixture_50_dict = rg.reg_state_fixed('y1', mixture_independent_vars, mixture)
y3_mixture_50_dict = rg.reg_state_fixed('y3', mixture_independent_vars, mixture)
# urban10
urban_10_independent_vars = urban_10[independent_vars_cols]
y1_urban_10_dict = rg.reg_state_fixed('y1', urban_10_independent_vars, urban_10)
y3_urban_10_dict = rg.reg_state_fixed('y3', urban_10_independent_vars, urban_10)

# Tables in Paper

## Table 1: Base Factors Regression

In [16]:
column_names = ["Avg Project Size", "Capacity Intensity", "Project Intensity"]
techno_base_table = create_regression_table(techno_base, column_names)

render_and_save_table(techno_base_table, 'latex/table_1_techno.tex')
techno_base_table

## Table 2: CI with traditional Social Factors and Electric Price Regression

In [17]:
# CI = technoeconomic base + additional social factors
techno_social_expanded_y2 = (
    techno_base_dict["y2"]["stargazer"].models
    + techno_electric_dict["y2"]["stargazer"].models
    + trad_social_dict["y2"]["stargazer"].models
    + social_expanded_dict["y2"]["stargazer"].models
)
column_names = ["Base", "Energy Prices", "Traditional Social", "Expanded Social"]
techno_social_expanded_table = create_regression_table(
    techno_social_expanded_y2, column_names, dependent_variable_name="Capacity Intensity"
)
render_and_save_table(techno_social_expanded_table, "latex/table_2_CI.tex")

techno_social_expanded_table

## Table 4 in SI

In [18]:
techno_social_expanded_y1 = (
    techno_base_dict["y1"]["stargazer"].models
    + techno_electric_dict["y1"]["stargazer"].models
    + trad_social_dict["y1"]["stargazer"].models
    + social_expanded_dict["y1"]["stargazer"].models
)
column_names = ["Base", "Energy Prices", "Traditional Social", "Expanded Social"]
techno_social_expanded_y1_table = create_regression_table(
    techno_social_expanded_y1, column_names, dependent_variable_name="Avg Project Size"
)
render_and_save_table(techno_social_expanded_y1_table, "latex/table_4SI_AP.tex")
techno_social_expanded_y1_table

## Table 5 in SI

In [19]:
# PI technoeconomic base + (TABLE 5 IN SI)
techno_social_expanded_y3 = (
    techno_base_dict["y3"]["stargazer"].models
    + techno_electric_dict["y3"]["stargazer"].models
    + trad_social_dict["y3"]["stargazer"].models
    + social_expanded_dict["y3"]["stargazer"].models
)
column_names = ["Base", "Energy Prices", "Traditional Social", "Expanded Social"]
techno_social_expanded_y3_table = create_regression_table(
    techno_social_expanded_y3, column_names, dependent_variable_name="Project Intensity"
)
render_and_save_table(techno_social_expanded_y3_table, "latex/table_5SI_PI.tex")
techno_social_expanded_y3_table

## Table 6 in SI*

In [20]:
incl_rooftop_table = create_regression_table(
    y1_rooftop_dict['stargazer'].models, None, dependent_variable_name="Roof Capacity Intensity"
)
render_and_save_table(incl_rooftop_table, 'latex/table_6SI_incl_rooftop.tex')
incl_rooftop_table

## Table 8 in SI

In [21]:
column_names = ['Base', 'Small','Medium', 'Large']
techno_base_size_y3_table = create_regression_table(
    techno_base_size, column_names, dependent_variable_name="Project Intensity"
)
render_and_save_table(techno_base_size_y3_table, 'latex/table_8SI_PI_noGDP.tex')
techno_base_size_y3_table

## Table 10 in SI

In [22]:
column_names = ['Base', 'Urban', 'Mixture', 'Rural']
hetero_popl_dens = (
    techno_base_dict["y3"]["stargazer"].models
    + y3_urban_50_dict["stargazer"].models
    + y3_mixture_50_dict["stargazer"].models
    + y3_urban_10_dict["stargazer"].models
)
hetero_popl_dens_table_y3 = create_regression_table(
    hetero_popl_dens, column_names, dependent_variable_name="Project Intensity"
)
render_and_save_table(hetero_popl_dens_table_y3, 'latex/table_10SI_urban_PI.tex')
hetero_popl_dens_table_y3

## Table 11 in SI

In [23]:
column_names = ['Base', 'Urban', 'Mixture', 'Rural']
hetero_popl_dens = (
    social_expanded_dict["y1"]["stargazer"].models
    + y1_urban_50_dict["stargazer"].models
    + y1_mixture_50_dict["stargazer"].models
    + y1_urban_10_dict["stargazer"].models
)
hetero_popl_dens_table_y2 = create_regression_table(
    hetero_popl_dens, column_names, dependent_variable_name="Average Project Size"
)
render_and_save_table(hetero_popl_dens_table_y2, 'latex/table_11SI_urban_AP.tex')
hetero_popl_dens_table_y2